# LMA Feature Analysis: Three-Tier Dance Motion Classification

This notebook analyzes Laban Movement Analysis (LMA) features extracted from WHAM skeleton data
to identify which movement qualities differentiate **artistic dance (Tier 1)**, **suggestive dance (Tier 2)**, and **explicit content (Tier 3)**.

## Pipeline
1. Videos → WHAM (3D skeleton extraction) → LMA feature extraction (55 features per frame)
2. Compute per-video summary statistics and derived features
3. Statistical comparison (Cohen's d effect size) between tiers
4. Visualization of separating features and the three-tier gradient

## Dataset
- **Tier 1 (Artistic):** K-pop choreography, street dance, breakdancing, hip-hop, fitness (1MILLION, RedBull BC One, etc.)
- **Tier 2 (Suggestive):** Twerking, dancehall, perreo/reggaeton, chair dance, belly dance, heels/floorwork, bachata
- **Tier 3 (Explicit):** NPDI pornography dataset

In [ ]:
import numpy as np
import os, glob, csv
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 6)
matplotlib.rcParams['font.size'] = 11

OUT_DIR = '../output/lma_tier_analysis'
print('Output dir:', os.path.abspath(OUT_DIR))

## 1. Load Data

In [ ]:
# Read tier labels from manifest
tier_map = {}
with open(os.path.join(OUT_DIR, 'manifest.csv')) as f:
    for row in csv.DictReader(f):
        tier_map[row['video_id']] = row['tier']

# Load all LMA feature dictionaries
all_data = {}
for lma_path in sorted(glob.glob(os.path.join(OUT_DIR, '*/lma_dict_id0.npy'))):
    vid_id = lma_path.split('/')[-2]
    tier = tier_map.get(vid_id, 'UNK')
    if tier == 'UNK':
        continue
    d = np.load(lma_path, allow_pickle=True).item()
    all_data[vid_id] = {'tier': tier, 'raw': d}

t1_ids = [k for k, v in all_data.items() if v['tier'] == 'T1']
t2_ids = [k for k, v in all_data.items() if v['tier'] == 'T2']
t3_ids = [k for k, v in all_data.items() if v['tier'] == 'T3']
print(f'Loaded {len(all_data)} videos: {len(t1_ids)} T1 (artistic), {len(t2_ids)} T2 (suggestive), {len(t3_ids)} T3 (explicit)')

In [ ]:
# Show all available LMA feature keys
sample = next(iter(all_data.values()))['raw']
print(f'{len(sample)} raw LMA features:')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape}')

## 2. Compute Derived Features

Based on the Gemini Deep Research taxonomy (Laban Movement Analysis framework):
- **Effort Flow:** Bound (controlled) vs Free (released) — CV captures bound↔free transitions
- **Effort Weight:** Strong (powerful) vs Light (yielding)
- **Effort Time:** Sudden (impactful) vs Sustained (lingering)
- **Effort Space:** Direct (goal-oriented) vs Indirect (diffuse)
- **Pelvis dominance:** Ratio of pelvis energy to total/extremity energy
- **Pelvis rhythmicity:** Autocorrelation of pelvis velocity

In [ ]:
def compute_features(d):
    """Compute summary statistics and derived features from raw LMA time series."""
    r = {}
    
    # --- Raw feature summary stats (mean, std, CV) ---
    for key in d:
        if hasattr(d[key], 'shape') and len(d[key].shape) == 1:
            r[f'{key}_mean'] = float(np.nanmean(d[key]))
            r[f'{key}_std'] = float(np.nanstd(d[key]))
            r[f'{key}_cv'] = float(np.nanstd(d[key]) / (np.nanmean(d[key]) + 1e-8))
    
    # --- Derived features mapped to Gemini LMA taxonomy ---
    
    # Pelvis energy as fraction of total body energy
    r['pelvis_energy_ratio'] = np.nanmean(d['PELVIS_KE']) / (np.nanmean(d['Effort_Weight_Global']) + 1e-8)
    
    # Pelvis velocity relative to extremities (high = pelvis-focused)
    ext_vel = np.nanmean([np.nanmean(d[k]) for k in
                          ['L_WRIST_vel', 'R_WRIST_vel', 'L_ANKLE_vel', 'R_ANKLE_vel']])
    r['pelvis_ext_ratio'] = np.nanmean(d['PELVIS_vel']) / (ext_vel + 1e-8)
    
    # Pelvis rhythmicity via autocorrelation (high = repetitive cyclic motion)
    pv = d['PELVIS_vel']
    if len(pv) > 10:
        pv_n = (pv - np.nanmean(pv)) / (np.nanstd(pv) + 1e-8)
        autocorrs = [np.corrcoef(pv_n[:-lag], pv_n[lag:])[0, 1]
                     for lag in range(2, min(6, len(pv_n)))]
        r['pelvis_rhythmicity'] = float(np.nanmean(autocorrs))
    else:
        r['pelvis_rhythmicity'] = 0.0
    
    # Flow variability (bound↔free transitions = "sensual release" cycles)
    r['flow_cv'] = float(np.nanstd(d['Effort_Flow_Global']) /
                         (np.nanmean(d['Effort_Flow_Global']) + 1e-8))
    
    # Weight variability
    r['weight_cv'] = float(np.nanstd(d['Effort_Weight_Global']) /
                           (np.nanmean(d['Effort_Weight_Global']) + 1e-8))
    
    # Time variability (low CV = more sustained = suggestive per Gemini)
    r['time_cv'] = float(np.nanstd(d['Effort_Time_Global']) /
                         (np.nanmean(d['Effort_Time_Global']) + 1e-8))
    
    # Arm-to-pelvis energy ratio (low = arms quiet, pelvis active = suggestive)
    arm_ke = np.nanmean([np.nanmean(d['L_WRIST_KE']), np.nanmean(d['R_WRIST_KE'])])
    r['arm_pelvis_ke_ratio'] = arm_ke / (np.nanmean(d['PELVIS_KE']) + 1e-8)
    
    return r

features = {}
for vid_id, info in all_data.items():
    features[vid_id] = {'tier': info['tier'], **compute_features(info['raw'])}

all_feat_names = sorted(set(k for v in features.values() for k in v if k != 'tier'))
print(f'Computed {len(all_feat_names)} features per video')

## 3. Three-Tier Statistical Comparison

### 3a. T1 vs T2 (Cohen's d)

In [ ]:
def cohens_d(a, b):
    """Compute Cohen's d effect size between two groups."""
    if not a or not b: return 0
    ps = np.sqrt((np.nanstd(a)**2 + np.nanstd(b)**2) / 2)
    return abs(np.nanmean(a) - np.nanmean(b)) / ps if ps > 1e-8 else 0

# Compute effect size for every feature (T1 vs T2)
results_12 = []
for feat in all_feat_names:
    t1_vals = [features[v][feat] for v in t1_ids if feat in features[v]]
    t2_vals = [features[v][feat] for v in t2_ids if feat in features[v]]
    if not t1_vals or not t2_vals:
        continue
    results_12.append({
        'feature': feat,
        't2_mean': np.nanmean(t2_vals), 't1_mean': np.nanmean(t1_vals),
        'cohens_d': cohens_d(t1_vals, t2_vals),
        'direction': 'T2 higher' if np.nanmean(t2_vals) > np.nanmean(t1_vals) else 'T2 lower',
        't1_vals': t1_vals, 't2_vals': t2_vals,
    })
results_12.sort(key=lambda x: -x['cohens_d'])

print(f"{'Feature':<40} {'T2 mean':>9} {'T1 mean':>9} {'Cohen d':>9} {'Direction'}")
print('=' * 85)
for r in results_12[:15]:
    print(f"{r['feature']:<40} {r['t2_mean']:>9.3f} {r['t1_mean']:>9.3f} {r['cohens_d']:>9.2f}  {r['direction']}")

### 3b. Three-Tier Feature Comparison (T1 vs T2 vs T3)

In [ ]:
# Three-tier comparison with pairwise effect sizes
key_feats = [
    ('L_WRIST_Accel_mean', 'Wrist Acceleration'),
    ('L_WRIST_KE_mean', 'Wrist Kinetic Energy'),
    ('Traj_Curvature_mean', 'Trajectory Curvature'),
    ('Traj_Path_Length_mean', 'Trajectory Path Length'),
    ('PELVIS_KE_cv', 'Pelvis KE Variability'),
    ('PELVIS_Directness_mean', 'Pelvis Directness'),
    ('PELVIS_vel_mean', 'Pelvis Velocity'),
    ('PELVIS_Jerk_mean', 'Pelvis Jerkiness'),
    ('Effort_Flow_Global_mean', 'Effort Flow'),
    ('Effort_Weight_Global_mean', 'Effort Weight'),
    ('Effort_Time_Global_mean', 'Effort Time'),
    ('flow_cv', 'Flow CV (bound-free)'),
    ('body_volume_mean', 'Body Volume'),
    ('pelvis_energy_ratio', 'Pelvis Energy Ratio'),
    ('arm_pelvis_ke_ratio', 'Arm/Pelvis KE Ratio'),
]

print(f"{'Feature':<28} {'T1 (artistic)':>14} {'T2 (suggestive)':>16} {'T3 (explicit)':>14}")
print('=' * 78)
for feat_key, label in key_feats:
    t1v = [features[v][feat_key] for v in t1_ids if feat_key in features[v]]
    t2v = [features[v][feat_key] for v in t2_ids if feat_key in features[v]]
    t3v = [features[v][feat_key] for v in t3_ids if feat_key in features[v]]
    print(f'{label:<28} {np.nanmean(t1v):>14.3f} {np.nanmean(t2v):>16.3f} {np.nanmean(t3v):>14.3f}')

print(f"\n\n{'Feature':<28} {'T1-T2':>8} {'T2-T3':>8} {'T1-T3':>8}  (Cohen's d)")
print('=' * 60)
for feat_key, label in key_feats:
    t1v = [features[v][feat_key] for v in t1_ids if feat_key in features[v]]
    t2v = [features[v][feat_key] for v in t2_ids if feat_key in features[v]]
    t3v = [features[v][feat_key] for v in t3_ids if feat_key in features[v]]
    d12, d23, d13 = cohens_d(t1v, t2v), cohens_d(t2v, t3v), cohens_d(t1v, t3v)
    marker = ' ***' if max(d12, d23, d13) > 0.8 else (' **' if max(d12, d23, d13) > 0.5 else '')
    print(f'{label:<28} {d12:>8.2f} {d23:>8.2f} {d13:>8.2f}{marker}')

### 3c. Three-Tier Gradient Visualization

In [ ]:
# Three-tier gradient plots for key features
gradient_feats = [
    ('Traj_Path_Length_mean', 'Trajectory Path Length', 'Whole-body movement decreases'),
    ('PELVIS_KE_cv', 'Pelvis KE Variability', 'Pelvis irregularity increases'),
    ('PELVIS_Directness_mean', 'Pelvis Directness', 'Pelvis becomes more indirect'),
    ('Effort_Time_Global_mean', 'Effort Time', 'Movement becomes more sustained'),
    ('Effort_Weight_Global_mean', 'Effort Weight', 'Movement becomes lighter'),
    ('flow_cv', 'Flow CV', 'More bound-free oscillation'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
tier_colors = {'T1': '#3498db', 'T2': '#e74c3c', 'T3': '#2c3e50'}
tier_labels = ['T1\n(Artistic)', 'T2\n(Suggestive)', 'T3\n(Explicit)']

for idx, (feat_key, label, interpretation) in enumerate(gradient_feats):
    ax = axes[idx // 3][idx % 3]
    
    t1v = [features[v][feat_key] for v in t1_ids if feat_key in features[v]]
    t2v = [features[v][feat_key] for v in t2_ids if feat_key in features[v]]
    t3v = [features[v][feat_key] for v in t3_ids if feat_key in features[v]]
    
    bp = ax.boxplot([t1v, t2v, t3v], labels=tier_labels, patch_artist=True, widths=0.6)
    for i, color in enumerate(['#3498db', '#e74c3c', '#2c3e50']):
        bp['boxes'][i].set_facecolor(color)
        bp['boxes'][i].set_alpha(0.5)
    
    ax.scatter([1]*len(t1v), t1v, color='#3498db', s=25, zorder=3, alpha=0.8)
    ax.scatter([2]*len(t2v), t2v, color='#e74c3c', s=25, zorder=3, alpha=0.8)
    ax.scatter([3]*len(t3v), t3v, color='#2c3e50', s=25, zorder=3, alpha=0.8)
    
    # Draw gradient line through means
    means = [np.nanmean(t1v), np.nanmean(t2v), np.nanmean(t3v)]
    ax.plot([1, 2, 3], means, 'k--', linewidth=1.5, alpha=0.5, zorder=4)
    
    ax.set_title(f'{label}\n({interpretation})', fontsize=9)

fig.suptitle('Three-Tier LMA Gradient: T1 (Athletic) → T2 (Suggestive) → T3 (Explicit)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'three_tier_gradient.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Top Separating Features — Visualization

In [ ]:
# Bar chart: top 15 features by effect size
top_n = 15
top = results[:top_n]

fig, ax = plt.subplots(figsize=(12, 6))
y_pos = np.arange(top_n)
colors = ['#e74c3c' if r['direction'] == 'T2 higher' else '#3498db' for r in top]
bars = ax.barh(y_pos, [r['cohens_d'] for r in top], color=colors, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels([r['feature'] for r in top])
ax.invert_yaxis()
ax.set_xlabel("Cohen's d (effect size)")
ax.set_title('Top 15 LMA Features Separating Tier 2 (Suggestive) from Tier 1 (Artistic)')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='Large effect (d=0.8)')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#3498db', label='T2 LOWER than T1'),
    Patch(facecolor='#e74c3c', label='T2 HIGHER than T1'),
], loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'top_features_effect_size.png'), dpi=150)
plt.show()

In [ ]:
# Box plots for the top 8 features
top_feats = [r['feature'] for r in results[:8]]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for idx, feat in enumerate(top_feats):
    ax = axes[idx // 4][idx % 4]
    r = results[idx]
    
    bp = ax.boxplot([r['t1_vals'], r['t2_vals']],
                    labels=['T1\n(Artistic)', 'T2\n(Suggestive)'],
                    patch_artist=True,
                    widths=0.6)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][1].set_facecolor('#e74c3c')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_alpha(0.6)
    
    # Overlay individual points
    ax.scatter([1]*len(r['t1_vals']), r['t1_vals'], color='#2c3e50', s=20, zorder=3, alpha=0.7)
    ax.scatter([2]*len(r['t2_vals']), r['t2_vals'], color='#c0392b', s=20, zorder=3, alpha=0.7)
    
    ax.set_title(f"{feat}\n(d={r['cohens_d']:.2f})", fontsize=9)
    ax.tick_params(labelsize=8)

fig.suptitle('Distribution of Top 8 Separating LMA Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'top_features_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

# Scatter plots with all three tiers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def scatter_tier(ax, feat_x, feat_y, xlabel, ylabel, title):
    for vid_id in t1_ids:
        ax.scatter(features[vid_id][feat_x], features[vid_id][feat_y],
                  color='#3498db', s=60, alpha=0.7, edgecolors='white', zorder=3)
    for vid_id in t2_ids:
        ax.scatter(features[vid_id][feat_x], features[vid_id][feat_y],
                  color='#e74c3c', s=60, alpha=0.7, edgecolors='white', zorder=3)
    for vid_id in t3_ids:
        ax.scatter(features[vid_id][feat_x], features[vid_id][feat_y],
                  color='#2c3e50', s=80, alpha=0.9, edgecolors='white', zorder=4, marker='D')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(['T1 (Artistic)', 'T2 (Suggestive)', 'T3 (Explicit)'])

scatter_tier(axes[0], 'L_WRIST_KE_mean', 'PELVIS_KE_cv',
             'Wrist Kinetic Energy', 'Pelvis KE Variability',
             'Arm Activity vs Pelvis Irregularity')

scatter_tier(axes[1], 'Traj_Path_Length_mean', 'PELVIS_Directness_mean',
             'Trajectory Path Length', 'Pelvis Directness (low=indirect)',
             'Whole-Body Movement vs Pelvis Directness')

scatter_tier(axes[2], 'Effort_Time_Global_mean', 'flow_cv',
             'Effort Time (low=sustained)', 'Flow CV (bound-free transitions)',
             'Effort Time vs Flow Variability')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'scatter_three_tier.png'), dpi=150)
plt.show()

In [ ]:
# Scatter plot: arm activity vs pelvis irregularity
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Wrist KE vs Pelvis KE CV
ax = axes[0]
for vid_id in t1_ids:
    f = features[vid_id]
    ax.scatter(f['L_WRIST_KE_mean'], f['PELVIS_KE_cv'],
              color='#3498db', s=60, alpha=0.7, edgecolors='white', zorder=3)
for vid_id in t2_ids:
    f = features[vid_id]
    ax.scatter(f['L_WRIST_KE_mean'], f['PELVIS_KE_cv'],
              color='#e74c3c', s=60, alpha=0.7, edgecolors='white', zorder=3)
ax.set_xlabel('Left Wrist Kinetic Energy (mean)')
ax.set_ylabel('Pelvis KE Coefficient of Variation')
ax.set_title('Arm Activity vs Pelvis Irregularity')
ax.legend(['T1 (Artistic)', 'T2 (Suggestive)'])

# Plot 2: Arm-pelvis KE ratio vs Pelvis Directness  
ax = axes[1]
for vid_id in t1_ids:
    f = features[vid_id]
    ax.scatter(f['arm_pelvis_ke_ratio'], f['PELVIS_Directness_mean'],
              color='#3498db', s=60, alpha=0.7, edgecolors='white', zorder=3)
for vid_id in t2_ids:
    f = features[vid_id]
    ax.scatter(f['arm_pelvis_ke_ratio'], f['PELVIS_Directness_mean'],
              color='#e74c3c', s=60, alpha=0.7, edgecolors='white', zorder=3)
ax.set_xlabel('Arm / Pelvis KE Ratio')
ax.set_ylabel('Pelvis Directness (low = indirect)')
ax.set_title('Arm-Pelvis Energy Ratio vs Pelvis Directness')
ax.legend(['T1 (Artistic)', 'T2 (Suggestive)'])

# Plot 3: Wrist acceleration vs Flow CV
ax = axes[2]
for vid_id in t1_ids:
    f = features[vid_id]
    ax.scatter(f['L_WRIST_Accel_mean'], f['flow_cv'],
              color='#3498db', s=60, alpha=0.7, edgecolors='white', zorder=3)
for vid_id in t2_ids:
    f = features[vid_id]
    ax.scatter(f['L_WRIST_Accel_mean'], f['flow_cv'],
              color='#e74c3c', s=60, alpha=0.7, edgecolors='white', zorder=3)
ax.set_xlabel('Left Wrist Acceleration (mean)')
ax.set_ylabel('Effort Flow CV (bound-free transitions)')
ax.set_title('Arm Activity vs Flow Variability')
ax.legend(['T1 (Artistic)', 'T2 (Suggestive)'])

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'scatter_tier_separation.png'), dpi=150)
plt.show()

## 6. Gemini Taxonomy Validation

The Gemini Deep Research taxonomy predicted these LMA markers for suggestive movement:

| LMA Factor | Predicted (Gemini) | Observed (Our Data) | Status |
|---|---|---|---|
| Flow | Free (released) | Higher flow CV (bound↔free oscillation) | Confirmed (nuanced) |
| Weight | Light (yielding) | Lower arm KE, lower overall weight | Confirmed |
| Space | Indirect (diffuse) | Lower pelvis directness | Confirmed |
| Pelvic isolation | High pelvis activity | Higher pelvis KE CV (irregular) | Confirmed |
| Time | Sustained (lingering) | Not a strong separator in raw form | Partially confirmed |

**New finding not in Gemini taxonomy:** The strongest signal is **arm inactivity** (d=1.37),
not pelvis activity. Suggestive dance concentrates energy in the pelvis while keeping
extremities quiet; athletic dance distributes energy across the whole body.

In [ ]:
# Gemini taxonomy features comparison table
taxonomy_features = [
    ('Effort_Flow_Global_cv', 'Flow variability (bound-free transitions)'),
    ('Effort_Weight_Global_mean', 'Effort Weight (strong vs light)'),
    ('Effort_Time_Global_mean', 'Effort Time (sudden vs sustained)'),
    ('Effort_Space_Global_mean', 'Effort Space (direct vs indirect)'),
    ('PELVIS_Directness_mean', 'Pelvis Directness'),
    ('PELVIS_KE_cv', 'Pelvis KE variability (isolation marker)'),
    ('L_WRIST_KE_mean', 'Left Wrist KE (arm activity)'),
    ('R_WRIST_KE_mean', 'Right Wrist KE (arm activity)'),
    ('arm_pelvis_ke_ratio', 'Arm/Pelvis energy ratio'),
    ('pelvis_ext_ratio', 'Pelvis/Extremity velocity ratio'),
    ('Traj_Curvature_mean', 'Trajectory curvature'),
    ('body_volume_mean', 'Body volume (compactness)'),
]

print(f"{'Description':<45} {'T2 mean':>9} {'T1 mean':>9} {'Ratio':>7} {'d':>6}")
print('=' * 82)
for feat_key, desc in taxonomy_features:
    match = [r for r in results if r['feature'] == feat_key]
    if match:
        r = match[0]
        ratio = r['t2_mean'] / (r['t1_mean'] + 1e-8)
        print(f"{desc:<45} {r['t2_mean']:>9.3f} {r['t1_mean']:>9.3f} {ratio:>6.2f}x {r['cohens_d']:>5.2f}")
    else:
        print(f"{desc:<45} {'(not found)':>9}")

# Heatmap of top features — all three tiers
top_feat_keys = [r['feature'] for r in results_12[:12]]
all_vids = t1_ids + t2_ids + t3_ids

matrix = np.array([[features[v].get(f, np.nan) for f in top_feat_keys] for v in all_vids])
matrix_z = (matrix - np.nanmean(matrix, axis=0)) / (np.nanstd(matrix, axis=0) + 1e-8)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(matrix_z, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5)

ax.set_xticks(range(len(top_feat_keys)))
ax.set_xticklabels([f.replace('_mean', '').replace('_', ' ')[:20] for f in top_feat_keys],
                    rotation=45, ha='right', fontsize=8)

labels = ['T1' for _ in t1_ids] + ['T2' for _ in t2_ids] + ['T3' for _ in t3_ids]
ax.set_yticks(range(len(all_vids)))
ax.set_yticklabels(labels, fontsize=8)

# Draw tier boundaries
ax.axhline(y=len(t1_ids) - 0.5, color='black', linewidth=2)
ax.axhline(y=len(t1_ids) + len(t2_ids) - 0.5, color='black', linewidth=2)

midpoints = [len(t1_ids)/2, len(t1_ids)+len(t2_ids)/2, len(t1_ids)+len(t2_ids)+len(t3_ids)/2]
tier_names = ['Tier 1\n(Artistic)', 'Tier 2\n(Suggestive)', 'Tier 3\n(Explicit)']
for mp, tn in zip(midpoints, tier_names):
    ax.text(-1.8, mp, tn, ha='center', va='center', fontweight='bold', fontsize=9)

plt.colorbar(im, ax=ax, label='Z-score')
ax.set_title('Top 12 Separating Features (Z-scored) — Three-Tier Gradient')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'feature_heatmap_3tier.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

### Three-Tier LMA Gradient

The tiers form a **monotonic ordinal gradient** — T2 is an intermediate state, T3 is an amplification:

| Signal | T1 (Athletic) | T2 (Suggestive) | T3 (Explicit) | Pattern |
|---|---|---|---|---|
| Trajectory path length | 4.12 | 2.21 | 0.74 | Less whole-body movement |
| Pelvis KE variability | 0.55 | 0.84 | 1.84 | Pelvis increasingly irregular |
| Pelvis directness | 28.0 | 19.4 | 8.9 | Pelvis increasingly indirect |
| Effort Time | 297 | 207 | 90 | More sustained, less sudden |
| Effort Weight | 37.6 | 25.2 | 8.1 | Movement becomes lighter |
| Flow CV | 0.35 | 0.47 | 0.60 | More bound-free oscillation |

### LMA Signatures
- **T1 (Athletic)** = active arms + structured direct whole-body motion + strong/sudden effort
- **T2 (Suggestive)** = quiet arms + irregular indirect pelvis + moderate flow variability
- **T3 (Explicit)** = near-stationary body + maximally irregular indirect pelvis + sustained light flow

### Key Findings
1. **Arm inactivity** is the strongest T1-vs-T2 signal (d=1.37), not pelvis activity
2. **Pelvis irregularity + overall stillness** is the T2-vs-T3 signal (d=1.5+)
3. **T1-vs-T3 effect sizes are massive** (d=2.0-2.9) — the endpoints are trivially separable
4. The **ordinal gradient** suggests an ordinal regression model may outperform standard multi-class classification
5. Gemini taxonomy **confirmed with nuance**: arm inactivity was the missing piece

## 8. Summary

### LMA Signature of Suggestive Dance (Tier 2)

**Primary signal — Arm inactivity (d > 1.0):**
- Left wrist acceleration 47% lower
- Left wrist kinetic energy 61% lower
- Trajectory curvature 43% lower
- Trajectory path length 46% lower

**Secondary signal — Pelvis irregularity (d ~ 0.8-1.0):**
- Pelvis KE coefficient of variation 52% higher
- Pelvis acceleration CV 49% higher
- Pelvis directness 31% lower (more indirect)

### Interpretation
Suggestive dance concentrates energy in the pelvis/hips with irregular, indirect motion
while keeping the arms relatively still. Athletic dance distributes energy across all limbs
with direct, structured trajectories.

This confirms the Gemini/LMA taxonomy (indirect + free flow + light weight) and adds
the new finding that **arm inactivity** is the single strongest differentiator — not pelvis
activity itself, which overlaps between tiers.